# 02_clean_audit

**Input:** `rides_combined_raw.parquet` (built by notebook 01)

**Output:** `rides_clean.parquet` (the cleaned rides) and `reports/cleaning_audit.csv` (rows removed at each step)

**Why this exists:** Investigates problems in the raw data, then applies five cleaning rules and logs how many rows each one removes, so every removal is accounted for.  The rules remove: duplicate rides (appear in two files), rides that started before Oct 1 2025, zero or negative ride length, rides over 24 hours (system-closed), and rides under 60 seconds (false starts)

In [1]:
# Where are the missing start stations, are the duplicates exact copies, how are ride lengths distributed, and what is the over-24h cluster

import pandas as pd
rides = pd.read_parquet("../data/clean/rides_combined_raw.parquet")
id_cols = [c for c in rides.columns if c != "source_file"]
rides["dur_min"] = (rides["ended_at"] - rides["started_at"]).dt.total_seconds() / 60
rides["start_null"] = rides["start_station_name"].isna()

# 1. Where are the missing start stations?
print(rides.groupby("rideable_type", observed=True)["start_null"].mean().round(3))
print(rides.groupby("member_casual", observed=True)["start_null"].mean().round(3))

# 2. Duplicates: exact copies or conflicting records?
dups = rides[rides["ride_id"].duplicated(keep=False)].sort_values("ride_id")
print("rows involved:", len(dups))
print("exact duplicate rows:", rides.duplicated(subset=id_cols).sum())
print("ride_ids spanning >1 file:", (dups.groupby("ride_id")["source_file"].nunique() > 1).sum())
print(dups[["ride_id", "rideable_type", "started_at", "ended_at", "member_casual", "source_file"]].head(10))

# 3. Duration bands and the over-24h cluster
bands = pd.cut(rides["dur_min"], [-float("inf"), 0, 1, 2, 5, 60, 360, 1440, float("inf")],
               labels=["<=0", "0-1", "1-2", "2-5", "5-60", "1-6h", "6-24h", ">24h"])
print(bands.value_counts().sort_index())
long = rides[rides["dur_min"] > 1440]
print(long[["rideable_type", "member_casual"]].value_counts())
print((long["dur_min"] / 60).describe())

# 4. Null end coords vs. over-24h overlap, and the window edge
endnull = rides["end_lat"].isna()
print("end coords null:", endnull.sum(), "| also over 24h:", (endnull & (rides["dur_min"] > 1440)).sum())
print("started before 2025-10-01:", (rides["started_at"] < "2025-10-01").sum())

rideable_type
classic_bike     0.000
electric_bike    0.294
Name: start_null, dtype: float64
member_casual
casual    0.218
member    0.204
Name: start_null, dtype: float64
rows involved: 70
exact duplicate rows: 35
ride_ids spanning >1 file: 35
                  ride_id  rideable_type              started_at  \
2043588  03C5375398BE0649  electric_bike 2026-04-30 23:46:36.039   
2784442  03C5375398BE0649  electric_bike 2026-04-30 23:46:36.039   
2164920  0612B2A21BA5E988   classic_bike 2026-04-30 15:56:11.305   
2507501  0612B2A21BA5E988   classic_bike 2026-04-30 15:56:11.305   
1984147  10117825A92C182D   classic_bike 2026-04-30 23:56:24.321   
2399509  10117825A92C182D   classic_bike 2026-04-30 23:56:24.321   
1916337  15521787039F37E5   classic_bike 2026-04-30 23:54:44.934   
2665751  15521787039F37E5   classic_bike 2026-04-30 23:54:44.934   
2175260  1DB04E901469FB2F  electric_bike 2026-04-30 23:54:09.726   
2864191  1DB04E901469FB2F  electric_bike 2026-04-30 23:54:09.726   

      

In [2]:
# Are rides under 60 seconds re-docks, what are the null end coordinates that are not over 24h, and do any over-24h rides have end coordinates

rides["dur_sec"] = (rides["ended_at"] - rides["started_at"]).dt.total_seconds()

# A. Sub-minute rides: re-docks, or something else?
short = rides[(rides["dur_sec"] > 0) & (rides["dur_sec"] < 60)]
s_id = short["start_station_id"].astype("string")
e_id = short["end_station_id"].astype("string")
both = s_id.notna() & e_id.notna()
print("sub-60s rides:", len(short))
print("same start/end station (both known):", round((s_id[both] == e_id[both]).mean(), 3), "of", both.sum())
print(short["member_casual"].value_counts(normalize=True).round(3))
print(short["rideable_type"].value_counts(normalize=True).round(3))

# B. Null end coordinates that are NOT over 24h
resid = rides[rides["end_lat"].isna() & (rides["dur_sec"] <= 86400)]
print("null end coords, not over 24h:", len(resid))
print(resid[["rideable_type", "member_casual"]].value_counts())
print((resid["dur_sec"] / 60).describe())

# C. Over-24h rides that DO have end coordinates
odd = rides[(rides["dur_sec"] > 86400) & rides["end_lat"].notna()]
print("over 24h with end coords:", len(odd))
print((odd["dur_sec"] / 3600).describe())

sub-60s rides: 159192
same start/end station (both known): 0.929 of 48450
member_casual
member    0.532
casual    0.468
Name: proportion, dtype: float64
rideable_type
electric_bike    1.0
classic_bike     0.0
Name: proportion, dtype: float64
null end coords, not over 24h: 308
rideable_type  member_casual
classic_bike   casual           208
               member           100
electric_bike  member             0
               casual             0
Name: count, dtype: int64
count     308.000000
mean      375.907825
std       469.913574
min         2.215950
25%        44.534187
50%       141.675592
75%       501.617962
max      1439.955050
Name: dur_sec, dtype: float64
over 24h with end coords: 181
count    181.000000
mean      24.668589
std        0.342624
min       24.001194
25%       24.361990
50%       24.754791
75%       24.997705
max       24.999633
Name: dur_sec, dtype: float64


In [3]:
# How far did the sub-minute rides actually travel; what share ended within 100 m of where they started

import numpy as np
short = rides[(rides["dur_sec"] > 0) & (rides["dur_sec"] < 60)]
lat0 = np.radians(short["start_lat"].astype("float64"))
dy = (short["end_lat"] - short["start_lat"]).astype("float64") * 111_000
dx = (short["end_lng"] - short["start_lng"]).astype("float64") * 111_000 * np.cos(lat0)
dist_m = np.hypot(dx, dy).dropna()
print(dist_m.describe())
print("within 100 m of start:", round((dist_m < 100).mean(), 3))

count    159192.000000
mean         78.493764
std         179.470223
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        8615.009211
dtype: float64
within 100 m of start: 0.802


In [4]:
# Are the negative-length rides a daylight-saving artifact?

neg = rides[rides["dur_sec"] < 0]
print("negative rides:", len(neg))
print(neg["started_at"].dt.date.value_counts())
print(neg["started_at"].dt.hour.value_counts())
fixed_min = (neg["dur_sec"] + 3600) / 60
print("if one hour is added back: shortest", round(fixed_min.min(), 1), "min, longest", round(fixed_min.max(), 1), "min")

negative rides: 29
started_at
2025-11-02    29
Name: count, dtype: int64
started_at
1    29
Name: count, dtype: int64
if one hour is added back: shortest 5.2 min, longest 47.5 min


In [5]:
# Apply the five rules in order, log rows removed at each step, save the cleaned rides and the audit table

rides = pd.read_parquet("../data/clean/rides_combined_raw.parquet")
audit = []
def log(name, before, df):
    audit.append({"step": name, "rows_before": before,
                  "rows_removed": before - len(df), "rows_after": len(df)})

n = len(rides)
rides = rides.drop_duplicates(subset=[c for c in rides.columns if c != "source_file"])
log("1. duplicate rides (appear in two files)", n, rides)

rides["dur_sec"] = (rides["ended_at"] - rides["started_at"]).dt.total_seconds()

n = len(rides)
rides = rides[rides["started_at"] >= "2025-10-01"]
log("2. started before Oct 1 2025", n, rides)

n = len(rides)
rides = rides[rides["dur_sec"] > 0]
log("3. zero or negative ride length", n, rides)

n = len(rides)
rides = rides[rides["dur_sec"] <= 86400]
log("4. over 24 hours (system-closed)", n, rides)

n = len(rides)
rides = rides[rides["dur_sec"] >= 60]
log("5. under 60 seconds (false starts)", n, rides)

rides.to_parquet("../data/clean/rides_clean.parquet")
pd.DataFrame(audit).to_csv("../reports/cleaning_audit.csv", index=False)
pd.DataFrame(audit)

,step,rows_before,rows_removed,rows_after
0,1. duplicate rides (appear in two files),6171785,35,6171750
1,2. started before Oct 1 2025,6171750,71,6171679
2,3. zero or negative ride length,6171679,29,6171650
3,4. over 24 hours (system-closed),6171650,5304,6166346
4,5. under 60 seconds (false starts),6166346,159192,6007154
